**CI twin of `ch07-gradient-descent-variants.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.25,
    random_state=42, stratify=digits.target)

with warnings.catch_warnings():          # 5 epochs — deliberately unconverged
    warnings.simplefilter("ignore")
    early = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                          random_state=0, max_iter=5).fit(Xtr, ytr)

G = early.predict_proba(Xtr) - np.eye(10)[ytr]   # p − onehot, row by row (Ch6)
full = G.mean(axis=0)                            # the full-batch verdict
print(f"full batch ({len(Xtr)} rows): db2[:4] = {np.round(full[:4], 4)}")

rng = np.random.default_rng(0)
for i in range(3):
    rows = rng.choice(len(Xtr), 32, replace=False)
    mb = G[rows].mean(axis=0)
    cos = mb @ full / (np.linalg.norm(mb) * np.linalg.norm(full))
    print(f"batch of 32: db2[:4] = {np.round(mb[:4], 4)}   agreement: {cos:.3f}")

print(f"cost per step: 32 rows instead of {len(Xtr)} — "
      f"{len(Xtr) / 32:.0f}× cheaper")

In [ ]:
steep, shallow = 10.0, 0.01

def ravine_loss(w):
    return 0.5 * (steep * w[0]**2 + shallow * w[1]**2)

def ravine_grad(w):
    return np.array([steep * w[0], shallow * w[1]])

w = np.array([1.0, 1.0])
for step in range(4):
    w = w - 0.21 * ravine_grad(w)
    print(f"step {step + 1}:  w1 = {w[0]:+.3f}   loss = {ravine_loss(w):.3f}")

In [ ]:
def descend(stepper, lr, max_steps=200000):
    """Walk until loss < 1e-3; return (steps, path)."""
    w = np.array([1.0, 1.0]); state = {}; path = [w.copy()]
    for t in range(1, max_steps + 1):
        w = stepper(w, ravine_grad(w), state, t, lr)
        path.append(w.copy())
        if ravine_loss(w) < 1e-3:
            return t, np.array(path)
    return None, np.array(path)

def gd_step(w, g, state, t, lr):
    return w - lr * g

n_gd, gd_path = descend(gd_step, lr=0.19)
print("w1 walk:", np.round(gd_path[:5, 0], 3), "…")
print(f"w2 after 5 steps: {gd_path[5, 1]:.4f}  (the crawl)")
print(f"steps to loss < 1e-3: {n_gd}")

In [ ]:
def momentum_step(w, g, state, t, lr, beta=0.9):
    state["v"] = beta * state.get("v", 0.0) + g
    return w - lr * state["v"]

w = np.array([1.0, 1.0]); v = np.zeros(2)
for step in range(1, 5):
    grad = ravine_grad(w)
    v = 0.9 * v + grad
    w = w - 0.19 * v
    print(f"step {step}:  g1 = {grad[0]:+.2f}  v1 = {v[0]:+.2f}    "
          f"g2 = {grad[1]:+.4f}  v2 = {v[1]:+.4f}")

n_mom, mom_path = descend(momentum_step, lr=0.19)
print(f"\nsteps to loss < 1e-3: {n_mom}  (plain GD: {n_gd})")

In [ ]:
def adam_walk(w, g, state, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    state["m"] = b1 * state.get("m", 0.0) + (1 - b1) * g
    state["v"] = b2 * state.get("v", 0.0) + (1 - b2) * g**2
    m_hat = state["m"] / (1 - b1**t)
    v_hat = state["v"] / (1 - b2**t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps)

n_adam, adam_path = descend(adam_walk, lr=0.19)
print(f"Adam, lr = 0.19: {n_adam} steps to loss < 1e-3\n")

print("        lr      GD   momentum   Adam")
for lr in [0.1, 0.01]:
    ng, _ = descend(gd_step, lr)
    nm, _ = descend(momentum_step, lr)
    na, _ = descend(adam_walk, lr)
    print(f"      {lr:4}   {ng:5d}   {nm:8d}   {na:4d}")

In [ ]:
import matplotlib.pyplot as plt

xs = np.linspace(-1.3, 1.3, 200); ys = np.linspace(-1.2, 1.2, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5 * (steep * XX**2 + shallow * YY**2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.contour(XX, YY, ZZ, levels=np.geomspace(5e-4, 6, 12),
           linewidths=0.6, colors="#94a3b8")
for path, label, colour in [
        (gd_path,   f"plain GD ({n_gd} steps)", "#dc2626"),
        (mom_path,  f"momentum ({n_mom} steps)", "#2563eb"),
        (adam_path, f"Adam ({n_adam} steps)",    "#059669")]:
    ax.plot(path[:, 0], path[:, 1], marker=".", markersize=3,
            linewidth=1, color=colour, label=label, alpha=0.8)
ax.scatter([0], [0], marker="*", s=140, color="black", zorder=5)
ax.set_xlabel("w1 (steep axis)"); ax.set_ylabel("w2 (shallow axis)")
ax.set_title("Same ravine, same start — three kinds of step")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
for name, kw in [
        ("plain SGD     ", dict(solver="sgd", momentum=0.0,
                                nesterovs_momentum=False)),
        ("SGD + momentum", dict(solver="sgd", momentum=0.9,
                                nesterovs_momentum=False)),
        ("Adam          ", dict(solver="adam"))]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        net = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                            random_state=0, max_iter=500, **kw).fit(Xtr, ytr)
    lc = net.loss_curve_
    under = next((i + 1 for i, l in enumerate(lc) if l < 0.1), None)
    print(f"{name}: epochs {net.n_iter_:3d}   final loss {net.loss_:.4f}   "
          f"test acc {net.score(Xte, yte):.3f}   "
          f"loss<0.1 at epoch {under}")

In [ ]:
w, v = 1.0, 0.0
for _ in range(2):
    g = w
    v = 0.4 * v + g
    w = w - 0.5 * v

run_tests([
    ("velocity after two steps", round(v, 4), 0.9),
    ("position after two steps", round(w, 4), 0.05),
])

In [ ]:
def adam_step(w, g, m, v, t, lr=0.1, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    return w - lr * m_hat / (v_hat**0.5 + eps), m, v

w1, m1, v1 = adam_step(1.0, 100.0, 0.0, 0.0, t=1)
w2, m2, v2 = adam_step(1.0, 0.0001, 0.0, 0.0, t=1)
w3, m3, v3 = adam_step(w1, 50.0, m1, v1, t=2)

run_tests([
    ("huge gradient, bounded first step", round(w1, 4), 0.9),
    ("tiny gradient, the SAME first step", round(w2, 4), 0.9),
    ("the ledgers chain into step two",
     [round(w3, 4), round(m3, 4), round(v3, 4)], [0.8068, 14.0, 12.49]),
])

In [ ]:
import numpy as np

def banana_loss(w):
    return (1 - w[0])**2 + 5 * (w[1] - w[0]**2)**2

def banana_grad(w):
    return np.array([-2 * (1 - w[0]) - 20 * w[0] * (w[1] - w[0]**2),
                     10 * (w[1] - w[0]**2)])

start = np.array([-1.0, 1.0])
print("start:", start, "  loss there:", banana_loss(start))